# CoStar San Francisco - Geographic Identifier Extraction


**Goal:** Extract and audit every geographic identifier in the dataset so each lease contract can be geocoded.  
**Market filter:** San Francisco only  
**Source:** `data/0_raw/CoStar_Whole.csv`

In [ ]:
import os
import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 100)

In [9]:
## 2. Data Loading — San Francisco only (chunked to avoid memory error)
import os
import pandas as pd

DATA_PATH = os.path.join("..", "..", "data", "0_raw", "CoStar_Whole.csv")

TARGET_MARKET = "San Francisco"
chunks = []

for chunk in pd.read_csv(
    DATA_PATH,
    encoding="latin1",
    on_bad_lines="skip",
    chunksize=500,
):
    filtered = chunk[chunk["Market"].astype(str).str.strip() == TARGET_MARKET]
    if not filtered.empty:
        chunks.append(filtered)

df = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()

print(f"Market filtered : {TARGET_MARKET}")
print(f"Loaded          : {df.shape[0]:,} rows  ×  {df.shape[1]:,} columns")
print(f"Memory          : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head(3)

Market filtered : San Francisco
Loaded          : 61,837 rows  ×  235 columns
Memory          : 363.5 MB


,LeaseCompId,AclId,AncillarySalesArea,BreakDate,BreakType,BuildOut,BuildOutStatus,BusinessRate,BusinessRatesYearly,DealType,...,UseableArea.RawValue,UseableArea.Code,Unnamed: 0,RentPlus.DisplayValue,RentPlus.Amount.DisplayValue,RentPlus.Amount.RawValue,RentPlus.Amount.Code,RentPlus.AreaCode,ReviewDate.DisplayValue,ReviewDate.RawValue
0,155701321.0,0.0,NaN,NaN,NaN,NaN,Partial Build-Out,NaN,NaN,New,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,163199091.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,111998375.0,0.0,NaN,NaN,NaN,NaN,Partial Build-Out,NaN,NaN,New,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
## Geographic column inventory — fill rates & sample values

GEO_COLS = [
    "LeaseCompId",           # unique lease ID (join key)
    "PropertyId",            # unique property ID
    "PropertyName",          # building name
    "Market",                # CoStar metro market
    "Submarket",             # CoStar submarket
    "Address.DeliveryAddress",  # street address
    "Address.Locality",      # city
    "Address.County",        # county
    "Address.PostalCode",    # ZIP / postal code
    "Address.RegionName",    # state (often blank — use Address.Subdivision)
    "Address.Subdivision",   # state abbreviation
    "Address.Country",       # country
]

present = [c for c in GEO_COLS if c in df.columns]
missing = [c for c in GEO_COLS if c not in df.columns]

summary = pd.DataFrame({
    "non_null"  : df[present].count(),
    "null_count": df[present].isna().sum(),
    "null_%"    : (df[present].isna().mean() * 100).round(1),
    "n_unique"  : df[present].nunique(),
    "sample"    : df[present].apply(lambda c: c.dropna().iloc[0] if c.notna().any() else "ALL NULL"),
})

print(f"Rows in SF subset : {len(df):,}")
print(f"Columns found     : {len(present)}")
if missing:
    print(f"Not in dataset    : {missing}")
print()
display(summary)

Rows in SF subset : 61,837
Columns found     : 12



,non_null,null_count,null_%,n_unique,sample
LeaseCompId,61837,0,0.0,61837,155701321.0
PropertyId,61837,0,0.0,9352,332739.0
PropertyName,25325,36512,59.0,1578,Perfect Peninsula Office Building
Market,61837,0,0.0,1,San Francisco
Submarket,61837,0,0.0,112,Brisbane/Daly City
Address.DeliveryAddress,61837,0,0.0,9196,333 Gellert Blvd
Address.Locality,61837,0,0.0,30,Daly City
Address.County,61837,0,0.0,2,San Mateo
Address.PostalCode,61813,24,0.0,59,94015.0
Address.RegionName,0,61837,100.0,0,ALL NULL


In [11]:
## Unique values for each geographic field

for col in present:
    vc = df[col].value_counts(dropna=False)
    n_unique = df[col].nunique()
    print(f"\n{'='*60}")
    print(f"  {col}  —  {n_unique} unique values  ({df[col].notna().sum():,} non-null)")
    print(f"{'='*60}")
    # Show all values if ≤ 50, otherwise top 30
    print(vc.head(50 if n_unique <= 50 else 30).to_string())


  LeaseCompId  —  61837 unique values  (61,837 non-null)
LeaseCompId
121579681.0    1
155701321.0    1
163199091.0    1
111998375.0    1
112014109.0    1
112055304.0    1
114130106.0    1
114141519.0    1
114141518.0    1
114141520.0    1
113577163.0    1
113565855.0    1
113561245.0    1
184479101.0    1
171028421.0    1
170069561.0    1
169825751.0    1
113722192.0    1
175244231.0    1
113811871.0    1
155350241.0    1
155455771.0    1
163196961.0    1
155424711.0    1
155424701.0    1
155424601.0    1
155424581.0    1
155454331.0    1
163380811.0    1
190079191.0    1

  PropertyId  —  9352 unique values  (61,837 non-null)
PropertyId
320717.0    421
320723.0    398
320724.0    379
333917.0    297
320736.0    283
320684.0    259
320469.0    258
320422.0    239
320420.0    225
332455.0    209
54387.0     207
320640.0    204
332462.0    203
320641.0    201
320952.0    194
320470.0    188
320472.0    186
320631.0    186
320647.0    174
332548.0    169
320635.0    169
320622.0    168
3

In [12]:
## Build geocodable address table

geo = df[present].copy()

# Construct a single full address string suitable for geocoding
geo["full_address"] = (
    geo.get("Address.DeliveryAddress", "").fillna("") + ", "
    + geo.get("Address.Locality", "").fillna("") + ", "
    + geo.get("Address.Subdivision", "").fillna("") + " "
    + geo.get("Address.PostalCode", "").astype(str).str.replace(r"\.0$", "", regex=True).str.strip() + ", "
    + geo.get("Address.Country", "").fillna("")
).str.replace(r"\s*,\s*,", ",", regex=True).str.strip(", ")

print(f"Total rows          : {len(geo):,}")
print(f"Rows with full addr : {geo['full_address'].str.len().gt(10).sum():,}")
print(f"Missing street addr : {geo['Address.DeliveryAddress'].isna().sum():,}")
print()
display(geo[["LeaseCompId","PropertyId","PropertyName","full_address",
             "Submarket","Address.PostalCode"]].head(20))

Total rows          : 61,837
Rows with full addr : 61,837
Missing street addr : 0



,LeaseCompId,PropertyId,PropertyName,full_address,Submarket,Address.PostalCode
0,155701321.0,332739.0,Perfect Peninsula Office Building,"333 Gellert Blvd, Daly City, CA 94015, United States",Brisbane/Daly City,94015.0
1,163199091.0,332735.0,NaN,"355 Gellert Blvd, Daly City, CA 94015, United States",Brisbane/Daly City,94015.0
2,111998375.0,332778.0,Century Plaza I,"1065 E Hillsdale Blvd, Foster City, CA 94404, United States",Foster City/Redwood Shrs,94404.0
3,112014109.0,332778.0,Century Plaza I,"1065 E Hillsdale Blvd, Foster City, CA 94404, United States",Foster City/Redwood Shrs,94404.0
4,112055304.0,332808.0,Metro Center Tower,"950 Tower Ln, Foster City, CA 94404, United States",Foster City/Redwood Shrs,94404.0
5,114130106.0,332427.0,NaN,"420-426 Valley Dr, Brisbane, CA 94005, United States",Brisbane/Daly City,94005.0
6,114141519.0,332563.0,NaN,"1 Edwards Ct, Burlingame, CA 94010, United States",Burlingame,94010.0
7,114141518.0,332563.0,NaN,"1 Edwards Ct, Burlingame, CA 94010, United States",Burlingame,94010.0
8,114141520.0,332460.0,Sea Breeze,"111 Anza Blvd, Burlingame, CA 94010, United States",Burlingame,94010.0
9,113577163.0,1579923.0,NaN,"1401-1405 Burlingame Ave, Burlingame, CA 94010, United States",Burlingame,94010.0


In [13]:
## Deduplicated property-level geocode table
# One row per unique PropertyId — use this as the geocoding input

prop_geo = (
    geo.dropna(subset=["Address.DeliveryAddress"])
    .drop_duplicates(subset=["PropertyId"])
    [["LeaseCompId", "PropertyId", "PropertyName", "full_address",
      "Address.DeliveryAddress", "Address.Locality",
      "Address.Subdivision", "Address.PostalCode",
      "Address.County", "Market", "Submarket"]]
    .sort_values("PropertyId")
    .reset_index(drop=True)
)

print(f"Unique properties with a street address: {len(prop_geo):,}")
display(prop_geo.head(20))

Unique properties with a street address: 9,352


,LeaseCompId,PropertyId,PropertyName,full_address,Address.DeliveryAddress,Address.Locality,Address.Subdivision,Address.PostalCode,Address.County,Market,Submarket
0,167100221.0,36156.0,NaN,"445 Bush St, San Francisco, CA 94108, United States",445 Bush St,San Francisco,CA,94108.0,San Francisco,San Francisco,Union Square
1,162545361.0,36157.0,New Montgomery Tower,"33 New Montgomery St, San Francisco, CA 94105, United States",33 New Montgomery St,San Francisco,CA,94105.0,San Francisco,San Francisco,South Financial District
2,70221222.0,36158.0,115 Sansome St,"115 Sansome St, San Francisco, CA 94104, United States",115 Sansome St,San Francisco,CA,94104.0,San Francisco,San Francisco,Financial District
3,133318191.0,36159.0,Second Street Square,"501 2nd St, San Francisco, CA 94107, United States",501 2nd St,San Francisco,CA,94107.0,San Francisco,San Francisco,Rincon/South Beach
4,70087748.0,36160.0,NaN,"110 Sutter St, San Francisco, CA 94104, United States",110 Sutter St,San Francisco,CA,94104.0,San Francisco,San Francisco,Financial District
5,204571941.0,36161.0,NaN,"1033-1045 Market St, San Francisco, CA 94103, United States",1033-1045 Market St,San Francisco,CA,94103.0,San Francisco,San Francisco,MidMarket
6,70054428.0,36162.0,NaN,"710 Market St, San Francisco, CA 94102, United States",710 Market St,San Francisco,CA,94102.0,San Francisco,San Francisco,Union Square
7,70017739.0,36163.0,Lululemon,"327-333 Grant Ave, San Francisco, CA 94108, United States",327-333 Grant Ave,San Francisco,CA,94108.0,San Francisco,San Francisco,Union Square
8,70225712.0,36164.0,NaN,"50 Mendell St, San Francisco, CA 94124, United States",50 Mendell St,San Francisco,CA,94124.0,San Francisco,San Francisco,Bayview/Hunters Point
9,10617690.0,36165.0,NaN,"1155 Mission St, San Francisco, CA 94103, United States",1155 Mission St,San Francisco,CA,94103.0,San Francisco,San Francisco,MidMarket


In [14]:
## Export geocode input file

OUT_PATH = os.path.join("..", "..", "data", "1_derived", "1_sf_properties_for_geocoding.csv")
prop_geo.to_csv(OUT_PATH, index=False)
print(f"Saved {len(prop_geo):,} rows → {OUT_PATH}")

Saved 9,352 rows → ..\..\data\1_derived\1_sf_properties_for_geocoding.csv
